<!-- GENERATED by scripts/gen_notebooks.py. Do not edit this cell. -->
# 1 · Text as Data

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/project-delphi/nlp-llms/blob/main/notebooks/01-text-as-data.ipynb)

**From Traditional NLP to Modern LLMs** · Day 1 · 95 minutes (45 briefing, 50 lab)

Tokenization, n-gram language models, TF-IDF and linear classifiers: the count-based baseline everything else is measured against.

**By the end of this notebook you can:**

- Tokenize text and justify the choices
- Build and evaluate an n-gram language model
- Train a linear text classifier and read its errors

**Stack:** NumPy, scikit-learn

In [ ]:
#@title Workshop harness: run this cell first { display-mode: "form" }
# Generated by scripts/gen_notebooks.py: do not edit this cell.
# Tick WORKED_EXAMPLE to run every reference solution in place of your code. That gives a
# complete worked example; it does not show that you did the exercises.
WORKED_EXAMPLE = False  #@param {type:"boolean"}

import os as _os
import time as _time

from IPython import get_ipython as _get_ipython

_NOTEBOOK = "01-text-as-data"
_CONTENT_SHA = "a30bcc4ed6cea7fe"
_EXERCISES = {1: ('tokenize', 'build_vocab'), 2: ('ngram_counts', 'prob'), 3: ('perplexity',), 4: ('sample',), 5: ('idf_weights', 'tfidf'), 6: ('prf_from_confusion',), 7: ('bm25_scores',)}
_HINTS = {2, 3, 5, 6}  # exercises with a folded Hint cell above their TODO cell
_PATHS = {'offline': 'offline stand-ins and test doubles: checks that the code runs, not what a model does', 'open': "the notebook's own code and models, no API keys", 'keyed': 'commercial APIs with keys'}
_FALLBACK_NOTE = None
_FALLBACK_CPU_ONLY = False  # the note describes the run a CPU runtime switches to
_RECORDED_SETTINGS = ('NLP_LLMS_OFFLINE_TINY', 'NLP_LLMS_STUB', 'NLP_LLMS_LAB07_OFFLINE', 'NLP_LLMS_LAB09_OFFLINE', 'NLP_LLMS_LAB10_OFFLINE', 'NLP_LLMS_LAB13_OFFLINE', 'NLP_LLMS_LAB14_OFFLINE', 'NLP_LLMS_LAB15_OFFLINE', 'NLP_LLMS_QUICK', 'HF_HUB_OFFLINE')
_OFFLINE_FLAGS = ('NLP_LLMS_OFFLINE_TINY', 'NLP_LLMS_STUB', 'NLP_LLMS_LAB07_OFFLINE', 'NLP_LLMS_LAB09_OFFLINE', 'NLP_LLMS_LAB10_OFFLINE', 'NLP_LLMS_LAB13_OFFLINE', 'NLP_LLMS_LAB14_OFFLINE', 'NLP_LLMS_LAB15_OFFLINE')
_FALLBACK_MARKERS = ('Could not load ', 'did not load', 'USING StubProvider', 'LSA stand-in fitted', 'OFFLINE TEST MODE')
_RECORD_PACKAGES = (
    "torch", "transformers", "numpy", "scikit-learn", "peft", "datasets", "openai",
    "anthropic", "typesafe-sdk", "langgraph", "langchain-core", "llama-index-core",
)


class _Workshop:
    """Keeps the folded solutions from replacing your code, and says whose code each
    checkpoint checked: yours, the reference solution, or code the lab provides."""

    _MISSING = object()

    def __init__(self, old=None):
        # Rerunning this cell (as Run all does) keeps the stored solutions but starts a new
        # run: the checkpoint results, cell times and run clock describe this run only.
        self.ref = getattr(old, "ref", {})
        self.stub = getattr(old, "stub", {})
        self.chosen = getattr(old, "chosen", set())  # exercises switched by use_reference
        self.verified = {}
        self._verifying = None
        self.results = {}
        self.cell_seconds = []
        self.cell_errors = 0
        self.fallbacks = set()
        self.started = _time.time()
        self._current = None
        self._t0 = None

    @property
    def worked(self):
        if _os.environ.get("NLP_LLMS_WORKED") == "1":
            return True
        return bool(_get_ipython().user_ns.get("WORKED_EXAMPLE"))

    def _is_reference(self, n, name, obj):
        return any(obj is r for r in self.ref.get(n, {}).get(name, ()))

    def solution(self, n):
        """Decorator on a reference solution: stores it and, unless WORKED_EXAMPLE is
        ticked, leaves your own definition of the same name in place."""

        def bind(obj, name=None):
            name = name or obj.__name__
            self.ref.setdefault(n, {}).setdefault(name, []).append(obj)
            current = _get_ipython().user_ns.get(name, self._MISSING)
            if current is not self._MISSING and not self._is_reference(n, name, current):
                self.stub.setdefault(n, {})[name] = current
            if self.worked:
                return obj
            if current is not self._MISSING and not self._is_reference(n, name, current):
                self.chosen.discard(n)  # your TODO cell ran again: back to your code
                return current
            if n in self.chosen:
                return obj  # you chose the reference with use_reference(n)
            # Your TODO cell has not run, or the name still holds a reference from a
            # worked run: go back to your code if it is known, never silently to the reference.
            yours = self.stub.get(n, {}).get(name, self._MISSING)
            if yours is not self._MISSING:
                return yours
            if callable(obj):
                return self._not_run(n, name)
            print(f"[workshop] Run your # TODO {n} cell: {name} is the reference until you do.")
            return obj

        return bind

    def _not_run(self, n, name):
        """Stands in for a function or class whose TODO cell has not run yet."""

        def missing(*args, **kwargs):
            raise NotImplementedError(
                f"{name}: run your # TODO {n} cell first, or go on with"
                f" workshop.use_reference({n!r})"
            )

        missing.__name__ = name
        return missing

    def solution_value(self, n, name, value):
        """The same as solution(n), for a value rather than a function or class."""
        if value is None or isinstance(value, (bool, int, float, str, bytes)):
            # Python shares equal small values, so yours and the reference could not be told
            # apart: give such a value in the stub instead of making it part of the exercise.
            kind = type(value).__name__
            raise TypeError(f"solution_value({n!r}, {name!r}): use an object, not a plain {kind}")
        return self.solution(n)(value, name)

    def use_reference(self, n):
        """Go on with the reference solution of exercise n in place of yours."""
        if n not in self.ref:
            raise KeyError(f"Exercise {n} has no reference solution yet: run its Solution cell")
        ns = _get_ipython().user_ns
        for name, objs in self.ref[n].items():
            ns[name] = objs[-1]
        self.chosen.add(n)
        print(
            f"Exercise {n}: now using the REFERENCE solution. Rerun the cells below it."
            f" To switch back to your code, rerun your # TODO {n} cell."
        )

    def checkpoint(self, n=None, label=None):
        """First line of a checkpoint cell: notes whose code the cell is about to check."""
        names = _EXERCISES.get(n, ())
        ns = _get_ipython().user_ns
        k = sum(self._is_reference(n, x, ns.get(x, self._MISSING)) for x in names)
        if not names:
            whose = None  # the results of the cells above: provided code, maybe your earlier work
        elif k == len(names):
            whose = "the REFERENCE solution"
        elif k == 0:
            whose = "your code"
        else:
            whose = "partly the REFERENCE solution"
        self._current = (n, str(label if label is not None else n), whose)

    def _pre(self, info):
        self._t0 = _time.time()

    def _post(self, result):
        if self._t0 is None:  # this harness cell itself: its hooks were registered mid-cell
            return
        self.cell_seconds.append(round(_time.time() - self._t0, 2))
        if not result.success and self._verifying is None:
            self.cell_errors += 1  # a verification copy fails on purpose; it is not an error
        if self._current is None:
            return
        n, label, whose = self._current
        self._current = None
        if self._verifying is not None:
            self.verified[self._verifying] = bool(result.success)
            return
        self.results[label] = (bool(result.success), whose)
        if whose is None:
            if result.success:
                print(f"[workshop] Checkpoint {label} passed.")
            elif isinstance(result.error_in_exec, NotImplementedError):
                print(
                    f"[workshop] Checkpoint {label} uses an exercise you have not written yet."
                    " Finish it, or go on with workshop.use_reference(N) for that exercise."
                )
            else:
                print(f"[workshop] Checkpoint {label} failed. Read the message above.")
        elif result.success:
            print(f"[workshop] Checkpoint {label} passed on {whose}.")
        elif isinstance(result.error_in_exec, NotImplementedError):
            print(
                f"[workshop] Checkpoint {label}: TODO {n} is not written yet. Write it, run"
                " its cell, then rerun this one. To go on without it:"
                f" workshop.use_reference({n!r})"
            )
            if n in _HINTS:
                print(f"[workshop] Stuck? Open the folded Hint above TODO {n} first.")
        elif whose == "the REFERENCE solution":
            print(
                f"[workshop] Checkpoint {label} failed on the reference solution: a problem"
                " with the lab or the runtime, not with your code. Tell the instructor."
            )
        else:
            print(
                f"[workshop] Checkpoint {label} failed on {whose}. Read the assertion above,"
                f" fix TODO {n}, or go on with workshop.use_reference({n!r})."
            )
            if n in _HINTS:
                print(f"[workshop] The folded Hint above TODO {n} says what to check.")

    def _verify_begin(self, n):
        """Test-only (scripts/test_notebooks.py --verify-checkpoints): bind exercise n's
        stubs, so that a copy of its checkpoint runs on unfinished code."""
        ns = _get_ipython().user_ns
        self._saved = {name: ns.get(name, self._MISSING) for name in _EXERCISES.get(n, ())}
        ns.update(self.stub.get(n, {}))
        self._verifying = n

    def _verify_end(self, n):
        """Test-only: restore the reference and require that the stub copy failed."""
        ns = _get_ipython().user_ns
        for name, value in self._saved.items():
            if value is self._MISSING:
                ns.pop(name, None)
            else:
                ns[name] = value
        self._verifying = None
        if self.verified.get(n) is not False:
            raise AssertionError(
                f"The checkpoint of exercise {n} passed on the unfinished stub: it cannot tell"
                " a finished exercise from an unfinished one"
            )

    def summary(self):
        """Which checkpoints passed, and on whose code."""
        groups = {"your code": [], "the reference": [], "other checks": [], "failed": []}
        for label, (ok, whose) in self.results.items():
            if not ok:
                groups["failed"].append(label)
            elif whose is None:
                groups["other checks"].append(label)
            elif whose == "your code":
                groups["your code"].append(label)
            else:
                groups["the reference"].append(label)
        mode = "WORKED EXAMPLE: reference solutions" if self.worked else "your code"
        print(f"{_NOTEBOOK}, run with {mode}.")
        path = self.path()
        # _PATHS comes from readiness.paths in _variables.yml; the offline entry says that
        # such numbers check the code, not a model.
        print(f"  What ran: {_PATHS[path]}.")
        # The labs choose their shorter CPU run from DEVICE, and Labs 7 and 10 let a participant
        # force the full run with FAST = False: a CPU-only note describes only the shorter run.
        ns = _get_ipython().user_ns
        short = ns["FAST"] if "FAST" in ns else not str(ns.get("DEVICE", "")).startswith("cuda")
        if path == "open" and _FALLBACK_NOTE and (short or not _FALLBACK_CPU_ONLY):
            print(f"  Read the numbers with this in mind: {_FALLBACK_NOTE}.")
        lines = {
            "your code": "Checkpoints passed on your code",
            "the reference": "Checkpoints passed on a reference solution",
            "other checks": "Other checks passed",
            "failed": "Checkpoints failed",
        }
        for group, labels in groups.items():
            print(f"  {lines[group]}: {', '.join(labels) or 'none'}")
        if self.worked:
            print("  A worked-example run shows how the lab goes, not that you did it.")

    def path(self):
        """offline, keyed or open: the same rule as scripts/add_run_record.py's path_of."""
        ns = _get_ipython().user_ns
        if self.fallbacks or any(_os.environ.get(flag) for flag in _OFFLINE_FLAGS):
            return "offline"
        if ns.get("PROVIDER") in ("openai", "anthropic") or ns.get("JEV_PATH") == "keyed":
            return "keyed"
        return "open"

    def run_record(self):
        """A run record for runs/ (see runs/README.md): paste it into
        scripts/add_run_record.py. It holds no code, outputs, keys or names."""
        import importlib.metadata as _md
        import json
        import platform
        import sys

        gpu = None
        try:
            import torch

            if torch.cuda.is_available():
                gpu = torch.cuda.get_device_name(0)
        except Exception:
            pass
        packages = {}
        for name in _RECORD_PACKAGES:
            try:
                packages[name] = _md.version(name)
            except Exception:
                pass
        import datetime as _dt

        record = {
            "date": _dt.datetime.now(_dt.timezone.utc).date().isoformat(),
            "notebook": _NOTEBOOK,
            "content_sha": _CONTENT_SHA,
            "mode": "worked" if self.worked else "learner",
            # A run passes when it reached this last cell with no error on the way and every
            # checkpoint passed. Run all reruns the harness first, so it starts clean.
            "status": "pass"
            if self.results and self.cell_errors == 0 and all(ok for ok, _ in self.results.values())
            else "fail",
            "cell_errors": self.cell_errors,
            "fallbacks": sorted(self.fallbacks),
            "seconds": round(_time.time() - self.started, 1),
            "cell_seconds": self.cell_seconds,
            "python": platform.python_version(),
            "platform": sys.platform,
            "gpu": gpu,
            "colab_release": _os.environ.get("COLAB_RELEASE_TAG"),
            "provider": _get_ipython().user_ns.get("PROVIDER"),
            "jev_path": _get_ipython().user_ns.get("JEV_PATH"),
            "settings": {k: _os.environ[k] for k in _RECORDED_SETTINGS if _os.environ.get(k)},
            "packages": packages,
        }
        print("----- run record (paste into scripts/add_run_record.py) -----")
        print(json.dumps(record))
        print("----- end of run record -----")


_ip = _get_ipython()
_old = _ip.user_ns.get("workshop")
# Remove only an earlier harness's own hooks; every other hook (Colab's, an extension's)
# stays registered.
for _event in ("pre_run_cell", "post_run_cell"):
    for _callback in list(_ip.events.callbacks[_event]):
        if type(getattr(_callback, "__self__", None)).__name__ == "_Workshop":
            _ip.events.unregister(_event, _callback)
workshop = _Workshop(_old if hasattr(_old, "ref") else None)

# Notice when a lab falls back to a stand-in, for the run record. print is wrapped once.
import builtins as _builtins

if not getattr(_builtins.print, "_workshop", False):
    _print = _builtins.print

    def _watching_print(*args, **kwargs):
        try:
            ip = _get_ipython()
            ws = ip.user_ns.get("workshop") if ip is not None else None
            if ws is not None and hasattr(ws, "fallbacks"):
                # Labs print their fallback notices as strings; never stringify other objects.
                text = " ".join(a for a in args if isinstance(a, str))
                ws.fallbacks.update(m.strip() for m in _FALLBACK_MARKERS if m in text)
        except Exception:
            pass  # watching must never break printing
        return _print(*args, **kwargs)

    _watching_print._workshop = True
    _builtins.print = _watching_print
_ip.events.register("pre_run_cell", workshop._pre)
_ip.events.register("post_run_cell", workshop._post)
print("Workshop harness ready:", "WORKED EXAMPLE" if workshop.worked else "checking your code")


## Setup

This lab runs on a CPU runtime. It needs no GPU, no API key and no installs: NumPy, SciPy, scikit-learn and matplotlib are preinstalled on Colab. Run all takes about a minute.

**How the lab works.** There are six exercises and one optional stretch section. Each exercise has:

1. a **Predict** question to answer before you run anything;
2. a `# TODO` cell where you write one or two short functions;
3. a folded **Solution** cell (open it only after trying);
4. a **Checkpoint** cell that tells you whether your function is right.

The folded solution cells never replace your code: each checkpoint checks your own functions and says so. If you are stuck, `workshop.use_reference(N)` lets you go on with exercise N's reference solution, and the checkpoint then says it checked the reference. To run the whole lab as a worked example, tick `WORKED_EXAMPLE` in the harness cell at the top; a worked run shows how the lab goes, not that you did it.

In [ ]:
# Setup: imports and seeds. Everything here is preinstalled on Colab.
import json
import math
import random
import re
import time
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import scipy.sparse as sp
import sklearn
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_recall_fscore_support,
)
from sklearn.naive_bayes import MultinomialNB

SEED = 0
random.seed(SEED)
np.random.seed(SEED)

NAVY, ACCENT = "#16324f", "#b3541e"  # plot colors
LAB_START = time.time()
print(f"NumPy {np.__version__} · scikit-learn {sklearn.__version__}")

## Data

Two datasets run through the whole workshop. This lab sets the baseline on both.

| | Language-model corpus | Classification set |
|---|---|---|
| Dataset | Tiny Shakespeare, 1,115,394 characters | arXiv Topics v1: titles and abstracts of arXiv papers |
| Task | Predict the next character | Predict the paper's primary category (4 classes) |
| Train / val / test | 1,000,000 / 55,000 / 60,394 characters, split by position | 4,800 / 600 / 1,600 papers, balanced, split stored in the file |
| Compared with | Lab 3 (LSTM) and Lab 5 (transformer) | Lab 2 (embeddings), Lab 6 (fine-tuned encoder), Lab 11 (calibration) |

The next cell is the workshop's standard loading cell, the same in every lab (see `data/README.md` in the repository). It checks each file against a hash, so everyone gets the same bytes and the same splits.

The rule for the whole workshop: fit on **train**, choose settings on **val**, report on **test** once per model.

In [ ]:
import csv, gzip, hashlib, io, os, urllib.request
from pathlib import Path


def fetch(name, urls, sha256):
    """Return the bytes of a workshop data file, checked against its SHA-256."""
    cache = Path(os.environ.get("NLP_LLMS_DATA", "data"))
    for path in (cache / name, Path("../data") / name):
        if path.exists() and hashlib.sha256(path.read_bytes()).hexdigest() == sha256:
            return path.read_bytes()
    for url in urls:  # canonical URL first, fallback second
        try:
            with urllib.request.urlopen(url, timeout=60) as response:
                blob = response.read()
        except OSError as error:
            print(f"Could not fetch {url}: {error}")
            continue
        if hashlib.sha256(blob).hexdigest() != sha256:
            print(f"Ignoring {url}: the file does not match the expected hash")
            continue
        cache.mkdir(parents=True, exist_ok=True)
        (cache / name).write_bytes(blob)
        return blob
    raise RuntimeError(f"Could not load {name} from any source")


TOPIC_CLASSES = ["cs.CL", "cs.CV", "cs.LG", "cs.RO"]  # label 0, 1, 2, 3
TOPIC_NAMES = ["Language", "Vision", "Machine learning", "Robotics"]


def load_topics():
    """arXiv Topics v1. Returns {"train" | "val" | "test": (texts, labels)}."""
    blob = fetch(
        "arxiv_topics_v1.csv.gz",
        [
            "https://raw.githubusercontent.com/project-delphi/nlp-llms/main/data/arxiv_topics_v1.csv.gz",
            "https://cdn.jsdelivr.net/gh/project-delphi/nlp-llms@main/data/arxiv_topics_v1.csv.gz",
        ],
        "49b51502fc6cf6ecff23fa266c5372594a88414dd96513579237792710246dc5",
    )
    splits = {"train": ([], []), "val": ([], []), "test": ([], [])}
    for row in csv.DictReader(io.StringIO(gzip.decompress(blob).decode("utf-8"))):
        texts, labels = splits[row["split"]]
        texts.append(row["title"] + "\n" + row["abstract"])
        labels.append(int(row["label"]))
    return splits


def load_lm_corpus():
    """Tiny Shakespeare. Returns {"train" | "val" | "test": str}, split by character offset."""
    blob = fetch(
        "tinyshakespeare.txt",
        [
            "https://raw.githubusercontent.com/karpathy/char-rnn/6f9487a6fe5b420b7ca9afb0d7c078e37c1d1b4e/data/tinyshakespeare/input.txt",
            "https://raw.githubusercontent.com/project-delphi/nlp-llms/main/data/tinyshakespeare.txt",
        ],
        "86c4e6aa9db7c042ec79f339dcb96d42b0075e16b8fc2e86bf0ca57e2dc565ed",
    )
    text = blob.decode("utf-8")
    return {"train": text[:1_000_000], "val": text[1_000_000:1_055_000], "test": text[1_055_000:]}

In [ ]:
corpus = load_lm_corpus()  # {"train" | "val" | "test": str}
topics = load_topics()  # {"train" | "val" | "test": (texts, labels)}

for split, text in corpus.items():
    print(f"LM corpus {split:<5} {len(text):>9,} characters")
for split, (texts, labels) in topics.items():
    print(f"Topics    {split:<5} {len(texts):>9,} papers, per class {np.bincount(labels).tolist()}")

print("\nStart of the LM corpus:\n" + corpus["train"][:180])
print("\nOne paper, class", TOPIC_NAMES[topics["train"][1][0]], "\n" + topics["train"][0][0][:300] + " ...")

# Part A · Language Modeling by Counting

## Exercise 1 · Tokens and Vocabulary (8 minutes)

A **tokenizer** splits a string into units called tokens. A **vocabulary** maps each token the model knows to an integer ID. Words below a count threshold are replaced by the unknown token `<unk>`.

**Predict.** The training split has about 12,000 distinct words. What fraction of them do you expect to occur exactly once? Write down a number.

**Task.** Complete the two functions.

- `tokenize(text)`: lowercase the text, then return a list in which each run of letters, digits or apostrophes is one token and every other non-space character is its own token. One `re.findall` call is enough.
- `build_vocab(tokens, min_count)`: return a dict from token to ID. `<unk>` is 0 and `<s>` (the start token) is 1. Then come the tokens whose count is at least `min_count`, most frequent first, ties in alphabetical order.

In [ ]:
# TODO 1: complete both functions.
def tokenize(text):
    """Lowercase `text`; return its words and punctuation marks as a list of strings."""
    raise NotImplementedError("TODO 1")


def build_vocab(tokens, min_count):
    """Return {token: id}: <unk> is 0, <s> is 1, then tokens with count >= min_count."""
    raise NotImplementedError("TODO 1")

In [ ]:
#@title Solution 1 — try it yourself first { display-mode: "form" }
@workshop.solution(1)
def tokenize(text):
    """Lowercase `text`; return its words and punctuation marks as a list of strings."""
    # runs of letters, digits or apostrophes, or any single non-space symbol
    return re.findall(r"[a-z0-9']+|[^a-z0-9'\s]", text.lower())


@workshop.solution(1)
def build_vocab(tokens, min_count):
    """Return {token: id}: <unk> is 0, <s> is 1, then tokens with count >= min_count."""
    vocab = {"<unk>": 0, "<s>": 1}
    counts = Counter(tokens)
    for token in sorted(counts, key=lambda t: (-counts[t], t)):  # frequent first, ties a-z
        if counts[token] >= min_count:
            vocab[token] = len(vocab)
    return vocab

<details>
<summary>Why this solution works</summary>

The regular expression has two alternatives. `[a-z0-9']+` takes the longest run of word characters, so `don't` stays one token. `[^a-z0-9'\s]` takes one character that is neither a word character nor whitespace, so each punctuation mark is its own token and whitespace is dropped. Lowercasing first merges *The* and *the*.

Assigning `len(vocab)` as the next ID keeps the IDs contiguous from 0 to |V| − 1, which is what lets a token index a row of a count table (and, from Module 2 on, a row of an embedding matrix). This is Section 2 of the briefing.

</details>

In [ ]:
# Checkpoint 1
workshop.checkpoint(1)
assert tokenize("Let's eat, Grandma!") == ["let's", "eat", ",", "grandma", "!"], (
    f'''got {tokenize("Let's eat, Grandma!")}, expected ["let's", "eat", ",", "grandma", "!"]: lowercase first, '''
    "keep the apostrophe inside a word, and make each punctuation mark its own token")
assert tokenize("ROMEO:\nO, speak again--3 times.") == [
    "romeo", ":", "o", ",", "speak", "again", "-", "-", "3", "times", ".",
], "punctuation marks must be separate tokens and whitespace dropped"
_v = build_vocab(["b", "a", "b", "c", "a", "b", "d"], min_count=2)
assert _v == {"<unk>": 0, "<s>": 1, "b": 2, "a": 3}, (
    f"got {_v}, expected <unk> 0, <s> 1, then b 2 (count 3) and a 3 (count 2): "
    "keep counts >= min_count, most frequent first, after the two special tokens")
_v = build_vocab(tokenize(corpus["val"]), min_count=1)
assert "<unk>" in _v and "<s>" in _v, f"<unk> and <s> must always be in the vocabulary; got {list(_v)[:5]}"
assert sorted(_v.values()) == list(range(len(_v))), "IDs must be contiguous from 0"
print("Checkpoint 1 passed")

**Run.** How much of the held-out text does the vocabulary cover? **Coverage** is the fraction of validation tokens that are in the vocabulary, that is, not mapped to `<unk>`.

In [ ]:
def encode(tokens, vocab):
    """Map tokens to IDs; anything not in the vocabulary becomes <unk>."""
    unk = vocab["<unk>"]
    return [vocab.get(token, unk) for token in tokens]


lm_tokens = {split: tokenize(text) for split, text in corpus.items()}
print(f"Word tokens: { {split: len(tokens) for split, tokens in lm_tokens.items()} }\n")
print("min_count  vocabulary size  coverage of val")
for min_count in (1, 2):
    vocab = build_vocab(lm_tokens["train"], min_count)
    coverage = np.mean([token in vocab for token in lm_tokens["val"]])
    print(f"{min_count:>9}  {len(vocab):>15,}  {coverage:>15.4f}")

In [ ]:
# Zipf's law: count against rank on log-log axes. Punctuation tokens are left out.
word_counts = Counter(t for t in lm_tokens["train"] if re.fullmatch(r"[a-z0-9']+", t))
freqs = np.array(sorted(word_counts.values(), reverse=True))
ranks = np.arange(1, len(freqs) + 1)
once = int((freqs == 1).sum())

fig, ax = plt.subplots(figsize=(6, 3.8))
ax.loglog(ranks, freqs, color=NAVY, linewidth=2, label="Tiny Shakespeare, train")
ax.loglog(ranks, freqs[0] / ranks, color=ACCENT, linestyle="--", label="slope −1")
ax.set_xlabel("rank r")
ax.set_ylabel("count f(r)")
ax.legend(frameon=False)
plt.show()

print("Most frequent words:", word_counts.most_common(5))
print(f"{once:,} of {len(freqs):,} distinct words ({once / len(freqs):.1%}) occur exactly once")
top10 = freqs[:10].sum() / freqs.sum()
print(f"The 10 most frequent words are {top10:.1%} of all word tokens")

**Explain.** Compare the printed fraction with your prediction. The curve is close to a straight line on log–log axes (Zipf's law), and nearly half of the vocabulary was seen once. Raising `min_count` from 1 to 2 removes almost half of the vocabulary and costs only about two points of coverage. Any estimate that depends on the count of one specific word is unreliable for most words; an estimate that depends on a specific *sequence* of words is worse, as the next exercises show.

**Justify the choice.** This tokenizer lowercases and splits off punctuation. Part B uses it for topic classification, while the baseline language model in Exercise 3 works on raw characters and keeps case, punctuation and newlines. In one sentence each, say why each choice suits its task (Section 2 of the briefing lists the trade-offs).

## Exercise 2 · Counting N-Grams and Add-k Smoothing (12 minutes)

An **n-gram model** predicts the next token from the previous $n - 1$ tokens, the **context**. Its parameters are two tables of counts from the training data:

- $c(\text{context}, w)$: how often token $w$ followed the context;
- $c(\text{context})$: how often the context occurred with any token after it.

The maximum-likelihood estimate is their ratio. **Add-k smoothing** adds $k$ to every count, so that nothing has probability zero:

$$
P_{\text{add-}k}(w \mid \text{context}) = \frac{c(\text{context}, w) + k}{c(\text{context}) + k\,|V|}.
$$

The functions work on lists of integer IDs, so the same code serves words and characters.

**Predict.** A context was never seen in training. What probability does the formula give each of the $|V|$ possible next tokens? Does it depend on $k$?

**Task.**

- `ngram_counts(ids, n, bos)`: pad the sequence on the left with `n - 1` copies of the start ID `bos`, then count. Return two `Counter`s: `grams`, keyed by the tuple `context + (w,)`, and `contexts`, keyed by the context tuple. For `n = 1` the context is the empty tuple `()`.
- `prob(w, context, counts, k, vocab_size)`: the formula above. `counts` is the pair returned by `ngram_counts`. A `Counter` returns 0 for a key it has not seen.

<details>
<summary>Hint for TODO 2</summary>

Adding $k$ to each of the $|V|$ possible next tokens raises the context's total by $k|V|$, not by $k$ (the add-k equation, Section 3 of the briefing). Check a context never seen in training: both counts are 0, so the formula must give $1/|V|$ for any $k > 0$; a denominator of $c(\text{context}) + k$ gives 1 instead. The context of position $t$ is the $n - 1$ IDs before $t$, never $t$ itself.

</details>

In [ ]:
# TODO 2: complete both functions.
def ngram_counts(ids, n, bos):
    """Count n-grams in a list of IDs. Returns (grams, contexts), two Counters."""
    grams, contexts = Counter(), Counter()
    padded = [bos] * (n - 1) + list(ids)
    for t in range(n - 1, len(padded)):
        ...  # the context is the n - 1 IDs before position t
    return grams, contexts


def prob(w, context, counts, k, vocab_size):
    """Add-k probability of token w after `context` (a tuple of n - 1 IDs)."""
    grams, contexts = counts
    ...

In [ ]:
#@title Solution 2 — try it yourself first { display-mode: "form" }
@workshop.solution(2)
def ngram_counts(ids, n, bos):
    """Count n-grams in a list of IDs. Returns (grams, contexts), two Counters."""
    grams, contexts = Counter(), Counter()
    padded = [bos] * (n - 1) + list(ids)
    for t in range(n - 1, len(padded)):
        context = tuple(padded[t - n + 1 : t])  # the n - 1 IDs before position t
        grams[context + (padded[t],)] += 1  # c(context, w)
        contexts[context] += 1  # c(context)
    return grams, contexts


@workshop.solution(2)
def prob(w, context, counts, k, vocab_size):
    """Add-k probability of token w after `context` (a tuple of n - 1 IDs)."""
    grams, contexts = counts
    return (grams[context + (w,)] + k) / (contexts[context] + k * vocab_size)

<details>
<summary>Why this solution works</summary>

`grams` and `contexts` are the two counts of the maximum-likelihood estimate (the MLE equation of the briefing, Section 3): the n-gram count and the context count. Because both are incremented in the same loop step, $c(\text{context}) = \sum_w c(\text{context}, w)$ exactly.

`prob` is add-k smoothing (the add-k equation of the briefing). Adding $k$ to each of the $|V|$ possible next tokens raises the context total by $k|V|$, so the probabilities still sum to one. For an unseen context both counts are 0 and the formula gives $k / (k|V|) = 1/|V|$, the uniform distribution, for any $k > 0$. No special case is needed.

</details>

In [ ]:
# Checkpoint 2
# Toy corpus "a b a b c" with a=0, b=1, c=2; the start ID 3 is outside the vocabulary.
workshop.checkpoint(2)
_toy = [0, 1, 0, 1, 2]
_grams, _contexts = _counts = ngram_counts(_toy, 2, bos=3)
assert _grams == Counter({(3, 0): 1, (0, 1): 2, (1, 0): 1, (1, 2): 1}), (
    f"bigram counts of 0 1 0 1 2 after the start ID 3: got {dict(_grams)}, expected "
    "{(3, 0): 1, (0, 1): 2, (1, 0): 1, (1, 2): 1}: key each count by the n - 1 IDs before t, plus the ID at t")
assert _contexts == Counter({(3,): 1, (0,): 2, (1,): 2}), (
    f"context counts: got {dict(_contexts)}, expected {{(3,): 1, (0,): 2, (1,): 2}}: "
    "count each context once per position it precedes")
assert ngram_counts(_toy, 1, bos=3)[1] == Counter({(): 5}), "for n = 1 the context is ()"

# Hand-computed, with k = 0.5 and |V| = 3:
assert math.isclose(prob(1, (0,), _counts, 0.5, 3), 2.5 / 3.5), (
    f"seen: P(b | a) = (2 + 0.5) / (2 + 1.5) = {2.5 / 3.5:.4f}, got {prob(1, (0,), _counts, 0.5, 3)}: "
    "the denominator is c(context) + k|V|, not c(context) + k")
assert math.isclose(prob(2, (0,), _counts, 0.5, 3), 0.5 / 3.5), "unseen: P(c | a) = 0.5 / (2 + 1.5)"

# The probabilities sum to 1 for a seen context, the start context and a never-seen context.
_tri = ngram_counts(_toy, 3, bos=3)
for _context in [(0, 1), (3, 3), (2, 2)]:
    _total = sum(prob(w, _context, _tri, 0.1, 3) for w in range(3))
    assert abs(_total - 1) < 1e-9, f"P(w | {_context}) sums to {_total} over w, not 1: is the denominator c(context) + k|V|?"
assert math.isclose(prob(0, (2, 2), _tri, 0.1, 3), 1 / 3), "unseen context must give 1 / |V|"
print("Checkpoint 2 passed")

## Exercise 3 · Perplexity (8 minutes)

To compare language models, ask how well each predicts text it has not seen. **Perplexity** is the exponential of the average negative log-probability per token (the **cross-entropy**, in nats because the log is natural):

$$
\mathrm{PPL} = \exp\Bigl(-\frac{1}{T}\sum_{t=1}^{T} \log P(w_t \mid \text{context}_t)\Bigr).
$$

Our three splits are consecutive pieces of one text. So the context of the first tokens of a split is real text: the end of the split before it. `perplexity` therefore takes a `history`, the $n - 1$ tokens that come immediately before `ids`. The history is context only: it is never predicted, and the average is over the $T$ tokens of `ids`. The helper `history_for` below builds it.

**Predict.** A model that knows nothing gives every token probability $1/|V|$. What is its perplexity?

**Task.** Complete `perplexity(ids, history, n, counts, k, vocab_size)`.

In [ ]:
def history_for(previous_ids, n, bos):
    """The n - 1 IDs just before a split: the end of `previous_ids`, left-padded with `bos`."""
    padded = [bos] * (n - 1) + list(previous_ids)
    return padded[len(padded) - (n - 1) :]


assert history_for([7, 8, 9], 3, bos=0) == [8, 9]
assert history_for([], 3, bos=0) == [0, 0]
assert history_for([7, 8, 9], 1, bos=0) == []

<details>
<summary>Hint for TODO 3</summary>

Divide by $T$, the number of predicted tokens, which is `len(ids)`: the history is context only, so it is neither predicted nor counted (the perplexity equation of the briefing). Use the natural log and exponentiate once, at the end. A model with no counts gives every token $1/|V|$, so its perplexity must be exactly $|V|$ for every $n$.

</details>

In [ ]:
# TODO 3: complete this function.
def perplexity(ids, history, n, counts, k, vocab_size):
    """Perplexity of the add-k n-gram model on `ids`; `history` holds the n - 1 IDs before it."""
    padded = list(history) + list(ids)  # position t of `padded` is predicted from the n - 1 before it
    ...

In [ ]:
#@title Solution 3 — try it yourself first { display-mode: "form" }
@workshop.solution(3)
def perplexity(ids, history, n, counts, k, vocab_size):
    """Perplexity of the add-k n-gram model on `ids`; `history` holds the n - 1 IDs before it."""
    padded = list(history) + list(ids)  # position t of `padded` is predicted from the n - 1 before it
    total = 0.0
    for t in range(n - 1, len(padded)):  # exactly len(ids) positions
        context = tuple(padded[t - n + 1 : t])
        total -= math.log(prob(padded[t], context, counts, k, vocab_size))
    return math.exp(total / len(ids))

<details>
<summary>Why this solution works</summary>

The loop is the perplexity equation of the briefing: sum $-\log P$ over the $T$ predicted positions, divide by $T$, exponentiate. The history supplies the context of the first $n - 1$ predictions and is not itself predicted, so the average is over `len(ids)` terms.

With no counts, add-k gives $1/|V|$ everywhere, the average negative log-probability is $\log |V|$, and the perplexity is $|V|$. That is why perplexity is read as an effective branching factor. Its logarithm is the cross-entropy loss that the LSTM of Lab 3 and the transformer of Lab 5 print during training.

</details>

In [ ]:
# Checkpoint 3
workshop.checkpoint(3)
_empty = (Counter(), Counter())
for _n in (1, 2, 4):
    _ppl = perplexity([0, 1, 2, 1, 0], history_for([], _n, bos=3), _n, _empty, 0.5, 3)
    assert abs(_ppl - 3) < 1e-6, (
        f"a model with no counts must have perplexity |V| = 3, got {_ppl} for n = {_n}: "
        "average over len(ids) predictions; the history is not predicted")
# One token: PPL = 1 / P(b | a) = 3.5 / 2.5 on the toy counts of Checkpoint 2.
_ppl = perplexity([1], [0], 2, _counts, 0.5, 3)
assert math.isclose(_ppl, 3.5 / 2.5), f"one token: PPL = 1 / P(b | a) = {3.5 / 2.5}, got {_ppl}"
# Two tokens "b c" after "a": the average is over 2 predictions, not 3.
_ppl = perplexity([1, 2], [0], 2, _counts, 0.5, 3)
assert math.isclose(_ppl, math.exp(-(math.log(2.5 / 3.5) + math.log(1.5 / 3.5)) / 2)), (
    f"two tokens after a history of one: expected {math.exp(-(math.log(2.5 / 3.5) + math.log(1.5 / 3.5)) / 2):.4f}, "
    f"got {_ppl}: divide by the 2 predicted tokens, not by the 3 positions of history + ids")
print("Checkpoint 3 passed")

**Run: why smoothing is needed.** Take a word-level bigram model with no smoothing ($k = 0$). One validation bigram that never occurred in training has probability 0, its log is $-\infty$, and the perplexity of the whole text is infinite. The cell counts how many there are.

In [ ]:
word_vocab = build_vocab(lm_tokens["train"], min_count=2)
WORD_V, WORD_BOS = len(word_vocab), word_vocab["<s>"]
word_ids = {split: encode(tokens, word_vocab) for split, tokens in lm_tokens.items()}

_grams, _ = ngram_counts(word_ids["train"], 2, WORD_BOS)
_padded = history_for(word_ids["train"], 2, WORD_BOS) + word_ids["val"]
zeros = sum(_grams[(_padded[t - 1], _padded[t])] == 0 for t in range(1, len(_padded)))
print(f"Word bigram model, k = 0: {zeros:,} of {len(word_ids['val']):,} validation tokens "
      f"({zeros / len(word_ids['val']):.1%}) have probability 0, so the perplexity is infinite.")

**Run: the character-level baseline.** This is the number Labs 3 and 5 are compared with, so the protocol is fixed:

- vocabulary: the 65 characters of the training split;
- counts from `train` only;
- $k$ chosen on `val` from the grid 0.001, 0.01, 0.1, 0.5, 1.0, separately for each order $n$;
- every character of the evaluated split is scored; the context of its first characters is the end of the preceding split.

**Predict.** Which order from 1 to 5 will have the lowest validation cross-entropy? Will the best $k$ grow or shrink as $n$ grows?

In [ ]:
chars = sorted(set(corpus["train"]))
char_to_id = {ch: i for i, ch in enumerate(chars)}
CHAR_V = len(chars)  # 65
CHAR_BOS = CHAR_V  # a start ID outside the vocabulary; it only pads the very start of train
assert CHAR_V == 65
char_ids = {split: [char_to_id[ch] for ch in text] for split, text in corpus.items()}

K_GRID = [0.001, 0.01, 0.1, 0.5, 1.0]
ORDERS = [1, 2, 3, 4, 5]

started = time.time()
char_counts = {n: ngram_counts(char_ids["train"], n, CHAR_BOS) for n in ORDERS}
char_val = {}  # (n, k) -> validation cross-entropy, nats per character
for n in ORDERS:
    history = history_for(char_ids["train"], n, CHAR_BOS)  # val follows train
    for k in K_GRID:
        ppl = perplexity(char_ids["val"], history, n, char_counts[n], k, CHAR_V)
        char_val[n, k] = math.log(ppl)

print("Validation cross-entropy (nats per character); * marks the best k for each n\n")
print(" n  distinct n-grams  " + "".join(f"k={k:<8}" for k in K_GRID))
char_best_k = {}
for n in ORDERS:
    char_best_k[n] = min(K_GRID, key=lambda k: char_val[n, k])
    row = "".join(f"{char_val[n, k]:.4f}{'*' if k == char_best_k[n] else ' '}   " for k in K_GRID)
    print(f"{n:>2}  {len(char_counts[n][0]):>16,}  {row}")
BEST_N = min(ORDERS, key=lambda n: char_val[n, char_best_k[n]])
print(f"\nBest order on val: n = {BEST_N} with k = {char_best_k[BEST_N]}   ({time.time() - started:.1f} s)")

In [ ]:
# Test, once per model: n = 2, n = 3 and the best order, each with the k chosen on val.
char_test = {}  # n -> test cross-entropy, nats per character
for n in sorted({2, 3, BEST_N}):
    history = history_for(char_ids["train"] + char_ids["val"], n, CHAR_BOS)  # test follows val
    ppl = perplexity(char_ids["test"], history, n, char_counts[n], char_best_k[n], CHAR_V)
    char_test[n] = math.log(ppl)

print(f"Uniform model: {math.log(CHAR_V):.4f} nats, perplexity {CHAR_V}\n")
print(" n      k  test nats/char  perplexity  bits/char")
for n, nll in char_test.items():
    print(f"{n:>2}  {char_best_k[n]:>5}  {nll:>14.4f}  {math.exp(nll):>10.4f}  {nll / math.log(2):>9.4f}")

**Explain.**

- Each added character of context lowers the validation cross-entropy, by less each time, and the best $k$ drops from 0.1 to 0.01. With longer contexts most counts are small, and adding even 0.1 to each of 65 possible next characters swamps them.
- At $n = 5$ the choice of $k$ matters more than the step from $n = 4$ to $n = 5$: look along the last row of the table.
- The number of distinct n-grams grows quickly with $n$ while the training text stays the same size. This is the sparsity problem.

**The baseline.** "Best character n-gram, $n \le 5$, add-k, test split": Lab 3 has to beat the last row of the table above.

In [ ]:
# Word level, for intuition only. NOT comparable with the character-level numbers
# or with any later lab: different tokens, different vocabulary, and <unk> absorbs rare words.
WORD_ORDERS = [1, 2, 3, 4]
word_counts_n = {n: ngram_counts(word_ids["train"], n, WORD_BOS) for n in WORD_ORDERS}
word_best_k, word_test = {}, {}
print(f"Word level, |V| = {WORD_V:,} (min_count = 2). Not comparable with later labs.\n")
print(" n  distinct n-grams  best k  val perplexity  test perplexity")
for n in WORD_ORDERS:
    history = history_for(word_ids["train"], n, WORD_BOS)
    val = {k: perplexity(word_ids["val"], history, n, word_counts_n[n], k, WORD_V) for k in K_GRID}
    word_best_k[n] = min(val, key=val.get)
    history = history_for(word_ids["train"] + word_ids["val"], n, WORD_BOS)
    word_test[n] = perplexity(
        word_ids["test"], history, n, word_counts_n[n], word_best_k[n], WORD_V
    )
    print(f"{n:>2}  {len(word_counts_n[n][0]):>16,}  {word_best_k[n]:>6}  "
          f"{val[word_best_k[n]]:>14.1f}  {word_test[n]:>15.1f}")

print("\nShare of test n-grams (raw words, no <unk>) that never occur in train:")
for n in (1, 2, 3, 4, 5):
    seen = set(zip(*[lm_tokens["train"][i:] for i in range(n)]))
    grams = list(zip(*[lm_tokens["test"][i:] for i in range(n)]))
    print(f"  n = {n}: {np.mean([g not in seen for g in grams]):.1%}")

At word level the trigram model is *worse* than the bigram model on held-out text. Most test trigrams never occurred in training, and add-k gives every unseen n-gram the same share, so the extra context adds noise instead of information. Better smoothing (the optional box in the briefing) helps; Module 2 attacks the cause.

## Exercise 4 · Sampling (5 minutes)

A language model is also a generator: draw the next token from $P(\cdot \mid \text{context})$, append it, slide the context forward, repeat.

**Predict.** How will samples from the $n = 2$ and $n = 4$ character models differ? What do you expect from $n = 1$?

**Task.** In `sample`, write the one line that draws the next ID from the distribution `p`. Use the generator `rng` (`rng.choice`) so that a seed reproduces the sample.

In [ ]:
# TODO 4: write the line that draws the next token.
def sample(history, length, n, counts, k, vocab_size, rng):
    """Generate `length` IDs after `history` (the n - 1 starting IDs) from the add-k n-gram model."""
    out = list(history)
    for _ in range(length):
        context = tuple(out[len(out) - (n - 1) :])
        p = np.array([prob(w, context, counts, k, vocab_size) for w in range(vocab_size)])
        p = p / p.sum()  # remove rounding error
        next_id = 0  # TODO: draw an ID from the distribution p with rng
        out.append(int(next_id))
    return out[len(history) :]

In [ ]:
#@title Solution 4 — try it yourself first { display-mode: "form" }
@workshop.solution(4)
def sample(history, length, n, counts, k, vocab_size, rng):
    """Generate `length` IDs after `history` (the n - 1 starting IDs) from the add-k n-gram model."""
    out = list(history)
    for _ in range(length):
        context = tuple(out[len(out) - (n - 1) :])
        p = np.array([prob(w, context, counts, k, vocab_size) for w in range(vocab_size)])  # (|V|,)
        p = p / p.sum()  # remove rounding error
        next_id = rng.choice(vocab_size, p=p)  # one draw from the categorical distribution p
        out.append(int(next_id))
    return out[len(history) :]

<details>
<summary>Why this solution works</summary>

`rng.choice(vocab_size, p=p)` returns ID $w$ with probability `p[w]`: one draw from the model's next-token distribution. Taking `argmax` instead would always pick the most likely token and soon loop. Every generator in this workshop, up to GPT, has this same loop; only the way `p` is computed changes.

</details>

In [ ]:
# Checkpoint 4
workshop.checkpoint(4)
_start = history_for(char_ids["train"][:100], 3, CHAR_BOS)
_a = sample(_start, 50, 3, char_counts[3], 0.1, CHAR_V, np.random.default_rng(1))
_b = sample(_start, 50, 3, char_counts[3], 0.1, CHAR_V, np.random.default_rng(1))
assert len(_a) == 50, "sample must return exactly `length` IDs"
assert all(isinstance(i, int) and 0 <= i < CHAR_V for i in _a), "IDs must be in the vocabulary"
assert _a == _b, "the same seed must give the same sample"
assert len(set(_a)) > 5, "the sample barely varies: are you drawing from p?"
print("Checkpoint 4 passed")

In [ ]:
rng = np.random.default_rng(SEED)
prompt = char_ids["train"][:4]  # "Firs"
print("CHARACTER LEVEL")
for n in ORDERS:
    ids = sample(history_for(prompt, n, CHAR_BOS), 200, n, char_counts[n], char_best_k[n], CHAR_V, rng)
    print(f"\n--- n = {n} ---\n" + "".join(chars[i] for i in ids))

n, tiny_k = 5, 1e-5
ids = sample(history_for(prompt, n, CHAR_BOS), 200, n, char_counts[n], tiny_k, CHAR_V, rng)
print(f"\n--- n = {n}, but with k = {tiny_k} ---\n" + "".join(chars[i] for i in ids))

id_to_word = {i: w for w, i in word_vocab.items()}
prompt = word_ids["train"][:3]  # "first citizen :"
print("\n\nWORD LEVEL")
for n in WORD_ORDERS:
    ids = sample(history_for(prompt, n, WORD_BOS), 40, n, word_counts_n[n], word_best_k[n], WORD_V, rng)
    print(f"\n--- n = {n} ---\n" + " ".join(id_to_word[i] for i in ids))

**Explain.**

- At $n = 1$ the characters have the right frequencies and nothing else. At $n = 3$ the fragments look like English, and the layout of a play appears.
- At $n = 4$ and $n = 5$, with the $k$ chosen on validation, the sample starts well and then collapses into random characters. This is add-k at work. A context seen once has count 1; with $k = 0.01$ the smoothing adds $0.01 \times 65 = 0.65$ to its total, so about 39% of the probability goes to the 64 characters that never followed it. One such draw creates a context that never occurred in training. There the distribution is uniform, the next context is unseen as well, and the sample never recovers. The word-level samples do the same from $n = 2$.
- The last character sample uses a much smaller $k$ and stays fluent. But the table in Exercise 3 shows the validation cross-entropy at $n = 5$ rising again when $k$ falls below 0.01. For this model, the setting that predicts held-out text best is not the one that generates the best-looking text.

Add-k treats every unseen continuation alike and has nowhere to go from an unseen context. Backoff and interpolation (the optional box in the briefing) fall back to a shorter context instead. The neural models of Modules 2 and 3 never meet an "unseen context" at all, because similar contexts get similar representations.

# Part B · Text Classification by Counting

The task changes: the input is a whole document (title and abstract of a paper) and the output is one of four classes.

## Exercise 5 · TF-IDF and Two Linear Classifiers (10 minutes)

The **bag-of-words** vector of a document has one entry per vocabulary word: the count of that word in the document. The next cell builds these vectors with your tokenizer from Exercise 1. The vocabulary rule: a word is kept if it occurs in at least 2 training documents (`min_df=2`).

**TF-IDF** reweights the counts by how rare each word is across documents. With $N$ training documents and $\mathrm{df}(w)$ the number of training documents that contain word $w$, scikit-learn's default is

$$
\mathrm{idf}(w) = \log\frac{1 + N}{1 + \mathrm{df}(w)} + 1, \qquad x_w = c(w, D)\cdot\mathrm{idf}(w), \qquad x \leftarrow \frac{x}{\lVert x \rVert_2}.
$$

**Predict.** Which token will have the lowest idf in this collection? Will a class name such as *robot* have a high or a low idf?

**Task.**

- `idf_weights(counts)`: the idf vector, shape `(|V|,)`, from the training count matrix, shape `(N, |V|)`.
- `tfidf(counts, idf)`: multiply each column by its idf, then scale each row to unit Euclidean length. Return a sparse matrix.

The matrices are SciPy sparse matrices. Useful pieces: `(counts > 0).sum(axis=0)`, `counts.multiply(idf)`, `x.multiply(x).sum(axis=1)`, `sp.diags(v) @ x`.

In [ ]:
train_texts, train_labels = topics["train"]
val_texts, val_labels = topics["val"]
test_texts, test_labels = topics["test"]
y_train, y_val, y_test = np.array(train_labels), np.array(val_labels), np.array(test_labels)

# Unigram counts with the Exercise 1 tokenizer (it lowercases, so the vectorizer must not).
vectorizer = CountVectorizer(tokenizer=tokenize, lowercase=False, token_pattern=None, min_df=2)
counts_train = vectorizer.fit_transform(train_texts)  # sparse (4800, |V|)
counts_val = vectorizer.transform(val_texts)  # sparse (600, |V|)
counts_test = vectorizer.transform(test_texts)  # sparse (1600, |V|)
feature_names = vectorizer.get_feature_names_out()
nonzero = counts_train.nnz / (counts_train.shape[0] * counts_train.shape[1])
print(f"Count matrix {counts_train.shape}, {nonzero:.2%} of its entries are nonzero")

<details>
<summary>Hint for TODO 5</summary>

Document frequency counts documents, not occurrences: a word used ten times in one abstract has $\mathrm{df} = 1$, so turn the counts into 0/1 before summing over the documents. The unit length is per row, one document each, so sum the squares along each row; normalizing the columns is the common slip (Section 4 of the briefing). The idf is fitted on the training matrix only.

</details>

In [ ]:
# TODO 5: complete both functions.
def idf_weights(counts):
    """idf(w) = log((1 + N) / (1 + df(w))) + 1 for each column of a sparse (N, |V|) count matrix."""
    raise NotImplementedError("TODO 5")


def tfidf(counts, idf):
    """Weight the columns of `counts` by `idf`, then scale each row to unit L2 norm. Sparse (N, |V|)."""
    raise NotImplementedError("TODO 5")

In [ ]:
#@title Solution 5 — try it yourself first { display-mode: "form" }
@workshop.solution(5)
def idf_weights(counts):
    """idf(w) = log((1 + N) / (1 + df(w))) + 1 for each column of a sparse (N, |V|) count matrix."""
    n_docs = counts.shape[0]
    df = np.asarray((counts > 0).sum(axis=0)).ravel()  # (|V|,) documents containing each word
    return np.log((1 + n_docs) / (1 + df)) + 1


@workshop.solution(5)
def tfidf(counts, idf):
    """Weight the columns of `counts` by `idf`, then scale each row to unit L2 norm. Sparse (N, |V|)."""
    x = sp.csr_matrix(counts.multiply(idf))  # (N, |V|), column w scaled by idf[w]
    norms = np.sqrt(np.asarray(x.multiply(x).sum(axis=1)).ravel())  # (N,)
    norms[norms == 0] = 1.0  # an empty document stays all-zero
    return sp.diags(1.0 / norms) @ x

<details>
<summary>Why this solution works</summary>

This is the scikit-learn TF-IDF equation of the briefing, Section 4. `(counts > 0).sum(axis=0)` counts documents, not occurrences: a word used ten times in one abstract still has document frequency 1. The idf comes from the **training** matrix only and is then applied to the validation and test counts, like any other fitted parameter.

Scaling each row to unit length stops a long abstract from having a larger vector than a short one. After it, the dot product of two rows is their cosine similarity.

</details>

In [ ]:
# Checkpoint 5
workshop.checkpoint(5)
_reference = TfidfTransformer().fit(counts_train)  # defaults: smooth_idf=True, norm="l2"
idf = idf_weights(counts_train)
assert idf.shape == (counts_train.shape[1],), (
    f"idf must have one entry per word, shape {(counts_train.shape[1],)}; got {idf.shape}: "
    "sum over the documents (axis 0) and flatten the result")
assert np.allclose(idf, _reference.idf_), (
    f"idf differs from scikit-learn's: first values {np.round(idf[:4], 4).tolist()}, expected "
    f"{np.round(_reference.idf_[:4], 4).tolist()}: df counts documents containing w, not occurrences")
X_train, X_val, X_test = tfidf(counts_train, idf), tfidf(counts_val, idf), tfidf(counts_test, idf)
assert sp.issparse(X_train), "return a sparse matrix"
for _ours, _counts_split in [(X_train, counts_train), (X_val, counts_val)]:
    assert np.allclose(_ours.toarray(), _reference.transform(_counts_split).toarray()), (
        "TF-IDF differs from TfidfTransformer; your first row norms are "
        f"{np.round(np.sqrt(np.asarray(_ours.multiply(_ours).sum(axis=1)).ravel()[:3]), 4).tolist()} "
        "(each should be 1): multiply column w by idf[w], then scale each row, not each column")
print("Checkpoint 5 passed: your TF-IDF matches TfidfTransformer")

_order = np.argsort(idf)
print("Lowest idf: ", [(feature_names[i], round(float(idf[i]), 2)) for i in _order[:6]])
print("idf of 'robot':", round(float(idf[vectorizer.vocabulary_["robot"]]), 2),
      "· highest possible (df = 2):", round(float(idf.max()), 2))

**Run: two linear classifiers.** Both compute one score per class as a weighted sum of the features. They differ in how the weights are set.

- **Logistic regression** learns the weights by minimizing cross-entropy on the TF-IDF vectors. Its L2 regularization strength is set through `C` (larger `C`, weaker penalty; this is scikit-learn's name and has nothing to do with the number of classes $C$ in the briefing), chosen here on the validation split by macro-F1 from a short grid.
- **Naive Bayes** reads the weights off the counts with add-one smoothing (`alpha=1.0`), exactly as in the briefing. It is fitted on the raw count matrix, since its model is about counts.

**Predict.** Which will do better on validation? By how much?

In [ ]:
C_GRID = [0.1, 1.0, 10.0, 100.0]

started = time.time()
models, val_scores = {}, {}
print("Logistic regression on TF-IDF\n     C  val accuracy  val macro-F1")
for C in C_GRID:
    models[C] = LogisticRegression(C=C, max_iter=1000, random_state=SEED).fit(X_train, y_train)
    pred = models[C].predict(X_val)
    val_scores[C] = (accuracy_score(y_val, pred), f1_score(y_val, pred, average="macro"))
    print(f"{C:>6}  {val_scores[C][0]:>12.4f}  {val_scores[C][1]:>12.4f}")
BEST_C = max(C_GRID, key=lambda C: val_scores[C][1])  # highest val macro-F1; ties go to the smaller C
clf = models[BEST_C]
print(f"Chosen C = {BEST_C}")

nb = MultinomialNB(alpha=1.0).fit(counts_train, y_train)
pred = nb.predict(counts_val)
nb_val = (accuracy_score(y_val, pred), f1_score(y_val, pred, average="macro"))
print(f"\nNaive Bayes on counts\n        {nb_val[0]:>12.4f}  {nb_val[1]:>12.4f}")
print(f"\n({time.time() - started:.1f} s)")

In [ ]:
# Softmax by hand: recompute predict_proba from the fitted weights.
W, b = clf.coef_, clf.intercept_  # W: (4, |V|) one row of word weights per class; b: (4,)
logits = X_val @ W.T + b  # (600, 4)  z = W x + b
logits = logits - logits.max(axis=1, keepdims=True)  # subtracting a constant per row changes nothing
probs = np.exp(logits) / np.exp(logits).sum(axis=1, keepdims=True)  # (600, 4) softmax
assert np.allclose(probs, clf.predict_proba(X_val), atol=1e-8)
assert (probs.argmax(axis=1) == clf.predict(X_val)).all()
print("predict_proba is softmax(W x + b). For the first validation paper:")
for name, p in zip(TOPIC_NAMES, probs[0]):
    print(f"  {name:<17} {p:.3f}")
print("True class:", TOPIC_NAMES[y_val[0]])

**Explain.** The two classifiers are close. Naive Bayes sets its weights in one pass over the counts; logistic regression optimizes them and can share weight between correlated words. On 4,800 training documents neither has a clear advantage. The fitted model is nothing more than a matrix `W` with one weight per class and word, and a bias `b`: the cell above reproduces its probabilities with three lines of NumPy (the softmax equation of the briefing, Section 5).

## Exercise 6 · Precision, Recall, F1 and Reading the Errors (7 minutes)

Accuracy is one number. The **confusion matrix** shows where the errors are: entry `cm[i, j]` counts test examples with true class `i` and predicted class `j`. For class $c$: true positives $\mathrm{TP}_c$ are the diagonal entry, false positives $\mathrm{FP}_c$ the rest of column $c$, false negatives $\mathrm{FN}_c$ the rest of row $c$.

$$
\mathrm{precision}_c = \frac{\mathrm{TP}_c}{\mathrm{TP}_c + \mathrm{FP}_c}, \qquad
\mathrm{recall}_c = \frac{\mathrm{TP}_c}{\mathrm{TP}_c + \mathrm{FN}_c}, \qquad
\mathrm{F1}_c = \frac{2\,\mathrm{precision}_c\,\mathrm{recall}_c}{\mathrm{precision}_c + \mathrm{recall}_c}.
$$

**Macro-F1** is the unweighted mean of $\mathrm{F1}_c$ over the classes.

`C` is now fixed, so this is the moment to predict on the test split, once.

**Predict.** Which two classes will be confused most often?

**Task.** Complete `prf_from_confusion(cm)`. No loops are needed: use `np.diag(cm)`, `cm.sum(axis=0)` and `cm.sum(axis=1)`.

In [ ]:
test_pred = clf.predict(X_test)  # the test split, used once, after C was chosen on val
cm = confusion_matrix(y_test, test_pred)  # (4, 4): rows are true classes, columns predicted
nb_test_pred = nb.predict(counts_test)

<details>
<summary>Hint for TODO 6</summary>

Rows of `cm` are true classes and columns are predictions. Precision asks how many cases predicted as $c$ were right, so its denominator is a column sum; recall asks how many true $c$ cases were found, so its denominator is a row sum. If your precision and recall look swapped against scikit-learn's, you swapped the axes (Section 6 of the briefing).

</details>

In [ ]:
# TODO 6: complete this function.
def prf_from_confusion(cm):
    """Return (precision, recall, f1, macro_f1): three arrays of shape (C,) and a float."""
    raise NotImplementedError("TODO 6")

In [ ]:
#@title Solution 6 — try it yourself first { display-mode: "form" }
@workshop.solution(6)
def prf_from_confusion(cm):
    """Return (precision, recall, f1, macro_f1): three arrays of shape (C,) and a float."""
    tp = np.diag(cm)  # (C,)
    precision = tp / cm.sum(axis=0)  # column sums: everything predicted as c
    recall = tp / cm.sum(axis=1)  # row sums: everything that truly is c
    f1 = 2 * precision * recall / (precision + recall)
    return precision, recall, f1, float(f1.mean())

<details>
<summary>Why this solution works</summary>

Column $c$ of the confusion matrix holds everything the model labeled $c$, so its sum is $\mathrm{TP}_c + \mathrm{FP}_c$, the denominator of precision. Row $c$ holds everything that truly is $c$, so its sum is $\mathrm{TP}_c + \mathrm{FN}_c$, the denominator of recall. These are the precision and recall equations of the briefing, Section 6. (If a class is never predicted its column sum is 0 and precision is undefined; that does not happen here.)

</details>

In [ ]:
# Checkpoint 6
workshop.checkpoint(6)
precision, recall, f1, macro_f1 = prf_from_confusion(cm)
_p, _r, _f, _ = precision_recall_fscore_support(y_test, test_pred)
assert np.allclose(precision, _p) and np.allclose(recall, _r) and np.allclose(f1, _f), (
    f"precision {np.round(precision, 3).tolist()} (expected {np.round(_p, 3).tolist()}), "
    f"recall {np.round(recall, 3).tolist()} (expected {np.round(_r, 3).tolist()}): "
    "precision divides by column sums (predicted c), recall by row sums (true c)")
assert math.isclose(macro_f1, f1_score(y_test, test_pred, average="macro")), (
    f"macro-F1 {macro_f1}, expected {f1_score(y_test, test_pred, average='macro'):.4f}: "
    "the unweighted mean of the per-class F1")
test_accuracy = float(np.trace(cm) / cm.sum())
print("Checkpoint 6 passed\n")

print(f"TF-IDF + logistic regression (C = {BEST_C}), test split, {cm.sum():,} papers")
print(f"accuracy {test_accuracy:.4f} · macro-F1 {macro_f1:.4f}\n")
print(f"{'class':<17} precision  recall      F1")
for name, p, r, f in zip(TOPIC_NAMES, precision, recall, f1):
    print(f"{name:<17} {p:>9.4f}  {r:>6.4f}  {f:>6.4f}")

In [ ]:
print("Confusion matrix (rows: true class, columns: predicted class)\n")
print(" " * 18 + "".join(f"{name[:8]:>10}" for name in TOPIC_NAMES))
for name, row in zip(TOPIC_NAMES, cm):
    print(f"{name:<18}" + "".join(f"{v:>10}" for v in row))

print("\nThe 10 highest-weight words per class")
for c, name in enumerate(TOPIC_NAMES):
    top = np.argsort(clf.coef_[c])[::-1][:10]
    print(f"  {name:<17} " + ", ".join(feature_names[top]))

print("\nThe 10 most confident errors on test (probability of the predicted class)")
test_probs = clf.predict_proba(X_test)  # (1600, 4)
wrong = np.flatnonzero(test_pred != y_test)
for i in wrong[np.argsort(test_probs[wrong].max(axis=1))[::-1][:10]]:
    title = test_texts[i].split("\n")[0]
    print(f"  {test_probs[i].max():.2f}  true {TOPIC_NAMES[y_test[i]]:<17} "
          f"predicted {TOPIC_NAMES[test_pred[i]]:<17} {title[:80]}")

**Explain.** Write two sentences in the cell below (double-click to edit):

1. Which pair of classes is confused most often, and in which direction?
2. Read the ten confident errors. Name one category of error you see: an ambiguous paper that fits both classes, a label that looks wrong, or a real limitation of a bag-of-words model.

*Your answer:*

Things to notice. The label is the authors' choice of primary category, and many papers are cross-listed, so some "errors" are papers that belong to both classes. The top-weight words are topic words; the model has no way to know that *LLM* and *language model* are related, or that a paper about *vision-language models for robots* mentions all three fields. That is where Module 2 starts.

## The Baseline Card

Every number below was measured in this run. Later labs compare against them:

- **Lab 3** (LSTM) and **Lab 5** (transformer): the character-level test cross-entropy, same splits, same 65 characters.
- **Lab 2** (averaged embeddings) and **Lab 6** (fine-tuned encoder): test accuracy and macro-F1 on the same 1,600 papers.

The cell also writes the card to `lab01_baselines.json` in the working directory. The values recorded in the repository (`data/baselines.json`) were produced by this cell.

In [ ]:
_nb_p, _nb_r, _nb_f, _ = precision_recall_fscore_support(y_test, nb_test_pred)
card = {
    "char_ngram": {
        str(n): {
            "k": char_best_k[n],
            "val_nats_per_char": round(char_val[n, char_best_k[n]], 4),
            "test_nats_per_char": round(nll, 4),
            "test_perplexity": round(math.exp(nll), 4),
            "test_bits_per_char": round(nll / math.log(2), 4),
        }
        for n, nll in char_test.items()
    },
    "char_ngram_best_n": BEST_N,
    "word_ngram_not_comparable": {
        str(n): {"k": word_best_k[n], "test_perplexity": round(word_test[n], 1)} for n in (2, 3)
    },
    "tfidf_logreg": {
        "C": BEST_C,
        "features": int(X_train.shape[1]),
        "val_accuracy": round(val_scores[BEST_C][0], 4),
        "val_macro_f1": round(val_scores[BEST_C][1], 4),
        "test_accuracy": round(test_accuracy, 4),
        "test_macro_f1": round(macro_f1, 4),
        "test_per_class": {
            TOPIC_CLASSES[c]: {
                "precision": round(float(precision[c]), 4),
                "recall": round(float(recall[c]), 4),
                "f1": round(float(f1[c]), 4),
            }
            for c in range(len(TOPIC_CLASSES))
        },
    },
    "counts_naive_bayes": {
        "alpha": 1.0,
        "val_accuracy": round(nb_val[0], 4),
        "val_macro_f1": round(nb_val[1], 4),
        "test_accuracy": round(accuracy_score(y_test, nb_test_pred), 4),
        "test_macro_f1": round(f1_score(y_test, nb_test_pred, average="macro"), 4),
        "test_per_class": {
            TOPIC_CLASSES[c]: {
                "precision": round(float(_nb_p[c]), 4),
                "recall": round(float(_nb_r[c]), 4),
                "f1": round(float(_nb_f[c]), 4),
            }
            for c in range(len(TOPIC_CLASSES))
        },
    },
    "versions": {"numpy": np.__version__, "scikit-learn": sklearn.__version__},
}
with open("lab01_baselines.json", "w", encoding="utf-8") as f:
    json.dump(card, f, indent=2)

print("BASELINE CARD · Lab 1 · measured in this run")
print("=" * 66)
print("Character n-gram LM, add-k, Tiny Shakespeare, test split (60,394 chars)")
print("   n      k   nats/char   perplexity   bits/char")
for n, row in card["char_ngram"].items():
    best = "  <- best n <= 5 (Lab 3 baseline)" if int(n) == BEST_N else ""
    print(f"  {n:>2}  {row['k']:>5}  {row['test_nats_per_char']:>10.4f}  "
          f"{row['test_perplexity']:>11.4f}  {row['test_bits_per_char']:>10.4f}{best}")
print("\nWord n-gram LM (min_count = 2), test perplexity. NOT comparable with later labs")
for n, row in card["word_ngram_not_comparable"].items():
    print(f"  {n:>2}  {row['k']:>5}  {row['test_perplexity']:>10.1f}")
print("\nClassifier, arXiv Topics v1, test split (1,600 papers)")
print("  model                                    accuracy   macro-F1")
row = card["tfidf_logreg"]
name = f"TF-IDF + logistic regression (C = {row['C']})"
print(f"  {name:<40} {row['test_accuracy']:>8.4f}   {row['test_macro_f1']:>8.4f}")
row = card["counts_naive_bayes"]
name = "counts + naive Bayes (alpha = 1.0)"
print(f"  {name:<40} {row['test_accuracy']:>8.4f}   {row['test_macro_f1']:>8.4f}")
print("=" * 66)
print(f"Run time so far: {time.time() - LAB_START:.0f} s")

In [ ]:
# Checkpoint: the baselines match the values recorded in data/baselines.json.
# The n-gram numbers are exact counts, so they must agree to rounding. The classifier is
# fitted by an iterative solver, so it gets a margin.
workshop.checkpoint(label="baselines")
RECORDED_NATS = {2: 2.4922, 3: 2.0859, 5: 1.8326}  # n -> test nats per character
assert BEST_N == 5, f"best order on val should be 5, got {BEST_N}"
for n, expected in RECORDED_NATS.items():
    assert abs(char_test[n] - expected) < 2e-3, (n, char_test[n], expected)
assert card["tfidf_logreg"]["test_accuracy"] >= 0.86, card["tfidf_logreg"]["test_accuracy"]
assert card["counts_naive_bayes"]["test_accuracy"] >= 0.86, card["counts_naive_bayes"]
print("Baselines reproduced. This is the end of the core path.")

## Stretch (optional) · BM25

For those who finish early. Nothing later in the workshop depends on this code; Module 13 uses BM25 again through a library.

TF-IDF can also rank documents for a search query. **BM25** is the refinement search engines have used for decades. For a query with word set $Q$ and a document $D$ of length $|D|$ tokens:

$$
\mathrm{score}(Q, D) = \sum_{w \in Q} \mathrm{idf}_{\text{BM25}}(w)\cdot
\frac{c(w, D)\,(k_1 + 1)}{c(w, D) + k_1\bigl(1 - b + b\,\frac{|D|}{\mathrm{avgdl}}\bigr)},
\qquad
\mathrm{idf}_{\text{BM25}}(w) = \log\Bigl(\frac{N - \mathrm{df}(w) + 0.5}{\mathrm{df}(w) + 0.5} + 1\Bigr).
$$

$\mathrm{avgdl}$ is the mean document length. $k_1$ controls how quickly repeated occurrences stop adding to the score (**saturation**), and $b$ controls how strongly long documents are penalized. Here $b$ is not a bias and $k_1$ is not the smoothing constant.

**Predict.** A document mentions the query word twice as often as another document of the same length. Is its BM25 score twice as high?

**Task.** Complete `bm25_scores(query_ids, counts, doc_lengths, k1, b)`. `query_ids` are column indices of the count matrix; return one score per document.

In [ ]:
# TODO 7 (stretch): complete this function.
def bm25_scores(query_ids, counts, doc_lengths, k1=1.5, b=0.75):
    """BM25 score of every document for a query. counts: sparse (N, |V|). Returns (N,)."""
    raise NotImplementedError("TODO 7")

In [ ]:
#@title Solution 7 — try it yourself first { display-mode: "form" }
@workshop.solution(7)
def bm25_scores(query_ids, counts, doc_lengths, k1=1.5, b=0.75):
    """BM25 score of every document for a query. counts: sparse (N, |V|). Returns (N,)."""
    counts = sp.csc_matrix(counts)  # fast column access
    n_docs = counts.shape[0]
    doc_lengths = np.asarray(doc_lengths, dtype=float)  # (N,)
    length_norm = k1 * (1 - b + b * doc_lengths / doc_lengths.mean())  # (N,)
    scores = np.zeros(n_docs)
    for w in set(query_ids):
        tf = counts[:, [w]].toarray().ravel()  # (N,) count of word w in each document
        df = (tf > 0).sum()
        idf = np.log((n_docs - df + 0.5) / (df + 0.5) + 1)  # never negative
        scores += idf * tf * (k1 + 1) / (tf + length_norm)
    return scores

<details>
<summary>Why this solution works</summary>

The fraction $\frac{c\,(k_1 + 1)}{c + \text{norm}}$ grows with the count $c$ but can never exceed $k_1 + 1$, so the tenth occurrence of a word adds much less than the first. The term `length_norm` is larger for documents longer than average, which lowers their score for the same count; with $b = 0$ it is the constant $k_1$ and length has no effect. The `+ 1` inside the logarithm keeps the idf non-negative even for a word found in most documents.

</details>

In [ ]:
# Checkpoint 7 (stretch)
workshop.checkpoint(7)
_m = sp.csr_matrix(np.array([[1, 0], [2, 0], [0, 3], [1, 0]]))  # 4 documents, 2 words
_same = bm25_scores([0], _m, doc_lengths=[10, 10, 10, 10])
assert _same.shape == (4,) and (_same >= 0).all(), "one non-negative score per document"
assert _same[2] == 0, "a document without the query word scores 0"
assert _same[0] < _same[1] < 2 * _same[0], "saturation: twice the count, less than twice the score"
_long = bm25_scores([0], _m, doc_lengths=[10, 10, 10, 40])
assert _long[3] < _long[0], "a longer document with the same count must score lower"
_b0 = bm25_scores([0], _m, doc_lengths=[10, 10, 10, 40], b=0.0)
assert math.isclose(_b0[3], _b0[0]), "with b = 0 the score must not depend on document length"
assert (bm25_scores(vectorizer.transform(["the of and"]).indices, counts_train,
                    np.asarray(counts_train.sum(axis=1)).ravel()) >= 0).all()
print("Checkpoint 7 passed")

In [ ]:
# Top 5 training documents for three queries: TF-IDF cosine against BM25.
doc_lengths = np.asarray(counts_train.sum(axis=1)).ravel()  # (4800,) tokens per document
for query in ["robot grasping", "speech recognition for low resource languages", "the learning"]:
    query_counts = vectorizer.transform([query])  # (1, |V|)
    cosine = (X_train @ tfidf(query_counts, idf).T).toarray().ravel()  # (4800,) rows are unit length
    bm25 = bm25_scores(query_counts.indices, counts_train, doc_lengths)
    print(f"\nQuery: {query!r}")
    for name, scores in [("TF-IDF cosine", cosine), ("BM25", bm25)]:
        print(f"  {name}")
        for i in np.argsort(scores)[::-1][:5]:
            print(f"    {scores[i]:>6.3f}  {len(train_texts[i].split()):>3} words  "
                  f"{train_texts[i].split(chr(10))[0][:70]}")

**Explain.** Compare the two rankings for each query. Where they differ, look at the document lengths: cosine similarity rewards a short document in which the query words are a large share of the vector, while BM25 saturates repeated words and applies a milder length penalty. For the last query, note what each method does with a word that appears in almost every document.

In [ ]:
# Generated by scripts/gen_notebooks.py: do not edit this cell.
# Which checkpoints passed, and on whose code; then a run record for the instructor.
workshop.summary()
workshop.run_record()

<!-- GENERATED by scripts/gen_notebooks.py. Do not edit this cell. -->
---
**Next:** [2 · Word Vectors and Neural Networks](https://colab.research.google.com/github/project-delphi/nlp-llms/blob/main/notebooks/02-word-vectors.ipynb)

[Workshop site](https://project-delphi.github.io/nlp-llms) · [All notebooks](https://project-delphi.github.io/nlp-llms/notebooks.html) · [Source](https://github.com/project-delphi/nlp-llms)

Genial Labs · text CC BY 4.0, code MIT